# SRP Checkpoint 2 - Izrada relacijskog modela i baze podataka

## 1. Predprocesiranje dataseta

### 1.1. Učitavanje potrebnih biblioteka

In [4]:
import pandas as pd

### 1.2. Učitavanje izvornog CSV skupa podataka

In [6]:
CSV_FILE_PATH = "Bank_Transaction_Fraud_Detection.csv"

df = pd.read_csv(CSV_FILE_PATH)

print("CSV size before:", df.shape)
df.head()

CSV size before: (200000, 24)


,Customer_ID,Customer_Name,Gender,Age,State,City,Bank_Branch,Account_Type,Transaction_ID,Transaction_Date,...,Merchant_Category,Account_Balance,Transaction_Device,Transaction_Location,Device_Type,Is_Fraud,Transaction_Currency,Customer_Contact,Transaction_Description,Customer_Email
0,d5f6ec07-d69e-4f47-b9b4-7c58ff17c19e,Osha Tella,Male,60,Kerala,Thiruvananthapuram,Thiruvananthapuram Branch,Savings,4fa3208f-9e23-42dc-b330-844829d0c12c,23-01-2025,...,Restaurant,74557.27,Voice Assistant,"Thiruvananthapuram, Kerala",POS,0,INR,+9198579XXXXXX,Bitcoin transaction,oshaXXXXX@XXXXX.com
1,7c14ad51-781a-4db9-b7bd-67439c175262,Hredhaan Khosla,Female,51,Maharashtra,Nashik,Nashik Branch,Business,c9de0c06-2c4c-40a9-97ed-3c7b8f97c79c,11-01-2025,...,Restaurant,74622.66,POS Mobile Device,"Nashik, Maharashtra",Desktop,0,INR,+9191074XXXXXX,Grocery delivery,hredhaanXXXX@XXXXXX.com
2,3a73a0e5-d4da-45aa-85f3-528413900a35,Ekani Nazareth,Male,20,Bihar,Bhagalpur,Bhagalpur Branch,Savings,e41c55f9-c016-4ff3-872b-cae72467c75c,25-01-2025,...,Groceries,66817.99,ATM,"Bhagalpur, Bihar",Desktop,0,INR,+9197745XXXXXX,Mutual fund investment,ekaniXXX@XXXXXX.com
3,7902f4ef-9050-4a79-857d-9c2ea3181940,Yamini Ramachandran,Female,57,Tamil Nadu,Chennai,Chennai Branch,Business,7f7ee11b-ff2c-45a3-802a-49bc47c02ecb,19-01-2025,...,Entertainment,58177.08,POS Mobile App,"Chennai, Tamil Nadu",Mobile,0,INR,+9195889XXXXXX,Food delivery,yaminiXXXXX@XXXXXXX.com
4,3a4bba70-d9a9-4c5f-8b92-1735fd8c19e9,Kritika Rege,Female,43,Punjab,Amritsar,Amritsar Branch,Savings,f8e6ac6f-81a1-4985-bf12-f60967d852ef,30-01-2025,...,Entertainment,16108.56,Virtual Card,"Amritsar, Punjab",Mobile,0,INR,+9195316XXXXXX,Debt repayment,kritikaXXXX@XXXXXX.com


### 1.3. Provjera strukture podataka

In [8]:
print("Veličina dataseta:", df.shape)

print("\nNazivi stupaca:")
print(df.columns.tolist())

print("\nTipovi podataka:")
print(df.dtypes)

print("\nNedostajuće vrijednosti:")
print(df.isnull().sum())

print("\nDuplicirani redci:")
print(df.duplicated().sum())

print("\nDuplicirani Transaction_ID:")
print(df["Transaction_ID"].duplicated().sum())

Veličina dataseta: (200000, 24)

Nazivi stupaca:
['Customer_ID', 'Customer_Name', 'Gender', 'Age', 'State', 'City', 'Bank_Branch', 'Account_Type', 'Transaction_ID', 'Transaction_Date', 'Transaction_Time', 'Transaction_Amount', 'Merchant_ID', 'Transaction_Type', 'Merchant_Category', 'Account_Balance', 'Transaction_Device', 'Transaction_Location', 'Device_Type', 'Is_Fraud', 'Transaction_Currency', 'Customer_Contact', 'Transaction_Description', 'Customer_Email']

Tipovi podataka:
Customer_ID                 object
Customer_Name               object
Gender                      object
Age                          int64
State                       object
City                        object
Bank_Branch                 object
Account_Type                object
Transaction_ID              object
Transaction_Date            object
Transaction_Time            object
Transaction_Amount         float64
Merchant_ID                 object
Transaction_Type            object
Merchant_Category           

### 1.4. Pretvorba datuma i vremena

In [10]:
df = df.drop_duplicates().copy()

df["Transaction_Date"] = pd.to_datetime(df["Transaction_Date"], format="%d-%m-%Y", errors="coerce")
df["Transaction_Time"] = pd.to_datetime(df["Transaction_Time"], format="%H:%M:%S", errors="coerce").dt.time

print("Veličina nakon predprocesiranja:", df.shape)

print("\nProvjera Transaction_Date i Transaction_Time:")
print(df[["Transaction_Date", "Transaction_Time"]].isnull().sum())

df.head()

Veličina nakon predprocesiranja: (200000, 24)

Provjera Transaction_Date i Transaction_Time:
Transaction_Date    0
Transaction_Time    0
dtype: int64


,Customer_ID,Customer_Name,Gender,Age,State,City,Bank_Branch,Account_Type,Transaction_ID,Transaction_Date,...,Merchant_Category,Account_Balance,Transaction_Device,Transaction_Location,Device_Type,Is_Fraud,Transaction_Currency,Customer_Contact,Transaction_Description,Customer_Email
0,d5f6ec07-d69e-4f47-b9b4-7c58ff17c19e,Osha Tella,Male,60,Kerala,Thiruvananthapuram,Thiruvananthapuram Branch,Savings,4fa3208f-9e23-42dc-b330-844829d0c12c,2025-01-23,...,Restaurant,74557.27,Voice Assistant,"Thiruvananthapuram, Kerala",POS,0,INR,+9198579XXXXXX,Bitcoin transaction,oshaXXXXX@XXXXX.com
1,7c14ad51-781a-4db9-b7bd-67439c175262,Hredhaan Khosla,Female,51,Maharashtra,Nashik,Nashik Branch,Business,c9de0c06-2c4c-40a9-97ed-3c7b8f97c79c,2025-01-11,...,Restaurant,74622.66,POS Mobile Device,"Nashik, Maharashtra",Desktop,0,INR,+9191074XXXXXX,Grocery delivery,hredhaanXXXX@XXXXXX.com
2,3a73a0e5-d4da-45aa-85f3-528413900a35,Ekani Nazareth,Male,20,Bihar,Bhagalpur,Bhagalpur Branch,Savings,e41c55f9-c016-4ff3-872b-cae72467c75c,2025-01-25,...,Groceries,66817.99,ATM,"Bhagalpur, Bihar",Desktop,0,INR,+9197745XXXXXX,Mutual fund investment,ekaniXXX@XXXXXX.com
3,7902f4ef-9050-4a79-857d-9c2ea3181940,Yamini Ramachandran,Female,57,Tamil Nadu,Chennai,Chennai Branch,Business,7f7ee11b-ff2c-45a3-802a-49bc47c02ecb,2025-01-19,...,Entertainment,58177.08,POS Mobile App,"Chennai, Tamil Nadu",Mobile,0,INR,+9195889XXXXXX,Food delivery,yaminiXXXXX@XXXXXXX.com
4,3a4bba70-d9a9-4c5f-8b92-1735fd8c19e9,Kritika Rege,Female,43,Punjab,Amritsar,Amritsar Branch,Savings,f8e6ac6f-81a1-4985-bf12-f60967d852ef,2025-01-30,...,Entertainment,16108.56,Virtual Card,"Amritsar, Punjab",Mobile,0,INR,+9195316XXXXXX,Debt repayment,kritikaXXXX@XXXXXX.com


### 1.5. Podjela 80/20

In [14]:
df80.to_csv("Bank_Transaction_Fraud_Detection_PROCESSED.csv", index=False)
df20.to_csv("Bank_Transaction_Fraud_Detection_PROCESSED_20.csv", index=False)

## 2. Definiranje entiteta, atributa, veza i kardinalnosti

### 2.1. Analiza jedinstvenih vrijednosti po stupcima

In [16]:
unique_counts = df80.nunique().sort_values(ascending=False)

print(unique_counts)

Customer_ID                160000
Transaction_ID             160000
Merchant_ID                160000
Transaction_Amount         158719
Account_Balance            158681
Customer_Name              121559
Transaction_Time            72885
Customer_Contact             9000
Customer_Email               4779
Transaction_Description       172
Transaction_Location          148
Bank_Branch                   145
City                          145
Age                            53
State                          34
Transaction_Date               30
Transaction_Device             20
Merchant_Category               6
Transaction_Type                5
Device_Type                     4
Account_Type                    3
Gender                          2
Is_Fraud                        2
Transaction_Currency            1
dtype: int64


### 2.2. Pregled vrijednosti po odabranim stupcima

In [18]:
for column in df80.columns:
    print("\n", column)
    print(df80[column].unique()[:10])


 Customer_ID
['d5f6ec07-d69e-4f47-b9b4-7c58ff17c19e'
 '3a73a0e5-d4da-45aa-85f3-528413900a35'
 '6c870d65-76b0-431d-bdf3-9292998e8211'
 '5323737c-bbd2-423f-9c9b-e0433c8f75dc'
 'c0c3d474-f6c2-4c66-9b0e-f9ba75c6f310'
 'e9a82764-1253-4a46-ad34-80e3416fc801'
 '708224d5-192a-4d86-b411-8ec6d1bb274b'
 '87b53799-9959-4534-9aad-f7dfc1821d4b'
 'd4d0f62e-00b0-4c0e-8188-0b5a3f3e6c3e'
 'b2563484-0dae-4555-8859-5b55f91136d4']

 Customer_Name
['Osha Tella' 'Ekani Nazareth' 'Ishanvi Dar' 'Arya Shroff' 'Jackson Shere'
 'Bhanumati Ravel' 'Meera Ganesh' 'Charvi Biswas' 'Yuvraj Dasgupta'
 'Amara Sidhu']

 Gender
['Male' 'Female']

 Age
[60 20 54 61 32 52 18 59 66 53]

 State
['Kerala' 'Bihar' 'Gujarat' 'Delhi' 'Andaman and Nicobar Islands'
 'Madhya Pradesh' 'Chhattisgarh' 'Punjab' 'Mizoram' 'Sikkim']

 City
['Thiruvananthapuram' 'Bhagalpur' 'Ahmedabad' 'New Delhi' 'Port Blair'
 'Bhopal' 'Jagdalpur' 'Vadodara' 'Chandigarh' 'Champhai']

 Bank_Branch
['Thiruvananthapuram Branch' 'Bhagalpur Branch' 'Ahmedabad 

### 2.3. Entiteti i atributi

| Entitet | Atributi |
|---|---|
| Customer | customer_id, customer_code, customer_name, gender, age, customer_contact, customer_email |
| State | state_id, state_name |
| City | city_id, city_name, state_id |
| Bank_Branch | bank_branch_id, bank_branch_name, city_id |
| Account_Type | account_type_id, account_type_name |
| Account | account_id, customer_id, account_type_id, bank_branch_id |
| Transaction_Type | transaction_type_id, transaction_type_name |
| Merchant_Category | merchant_category_id, merchant_category_name |
| Merchant | merchant_id, merchant_code, merchant_category_id |
| Device_Type | device_type_id, device_type_name |
| Transaction_Device | transaction_device_id, transaction_device_name, device_type_id |
| Transaction_Location | transaction_location_id, transaction_location_name |
| Currency | currency_id, currency_code |
| Bank_Transaction | transaction_id, transaction_code, account_id, merchant_id, transaction_type_id, transaction_device_id, transaction_location_id, currency_id, transaction_date, transaction_time, transaction_amount, account_balance, is_fraud, transaction_description |

### 2.4. Veze i kardinalnosti

| Veza | Kardinalnost | Objašnjenje |
|---|---|---|
| State - City | 1 : N | Jedna savezna država može imati više gradova, a jedan grad pripada jednoj saveznoj državi. |
| City - Bank_Branch | 1 : N | Jedan grad može imati više bankovnih poslovnica, a jedna poslovnica nalazi se u jednom gradu. |
| Customer - Account | 1 : N | Jedan korisnik može imati više računa, a jedan račun pripada jednom korisniku. |
| Account_Type - Account | 1 : N | Jedan tip računa može se pojaviti kod više računa, a jedan račun ima jedan tip računa. |
| Bank_Branch - Account | 1 : N | Jedna poslovnica može voditi više računa, a jedan račun pripada jednoj poslovnici. |
| Account - Bank_Transaction | 1 : N | Jedan račun može imati više transakcija, a jedna transakcija pripada jednom računu. |
| Transaction_Type - Bank_Transaction | 1 : N | Jedan tip transakcije može se pojaviti u više transakcija, a jedna transakcija ima jedan tip. |
| Merchant_Category - Merchant | 1 : N | Jedna kategorija trgovca može sadržavati više trgovaca, a jedan trgovac pripada jednoj kategoriji. |
| Merchant - Bank_Transaction | 1 : N | Jedan trgovac može biti povezan s više transakcija, a jedna transakcija se odnosi na jednog trgovca. |
| Device_Type - Transaction_Device | 1 : N | Jedan tip uređaja može obuhvaćati više uređaja, a jedan uređaj pripada jednom tipu uređaja. |
| Transaction_Device - Bank_Transaction | 1 : N | Jedan uređaj može biti korišten u više transakcija, a jedna transakcija koristi jedan uređaj. |
| Transaction_Location - Bank_Transaction | 1 : N | Jedna lokacija može se pojaviti u više transakcija, a jedna transakcija ima jednu lokaciju. |
| Currency - Bank_Transaction | 1 : N | Jedna valuta može se koristiti u više transakcija, a jedna transakcija ima jednu valutu. |

## 3. ER dijagram - konceptualni model

<img src="ER_dijagram.png" alt="ER dijagram - konceptualni model" width="800">

## 4. EER dijagram - logički model

### 4.1. Spajanje na MySQL

In [20]:
!pip install sqlalchemy pymysql

In [22]:
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

DB_USER = "root"
DB_PASSWORD = "root"
DB_HOST = "localhost"
DB_PORT = 3306
DB_NAME = "bank_fraud_db"

connection_string = f"mysql+pymysql://{DB_USER}:{quote_plus(DB_PASSWORD)}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine = create_engine(connection_string)

with engine.connect() as connection:
    result = connection.execute(text("SELECT DATABASE();"))
    print("Spojeno na bazu:", result.fetchone()[0])

Spojeno na bazu: bank_fraud_db


### 4.2. Definiranje i kreiranje tablica u MySQL bazi

In [24]:
from sqlalchemy import Column, Integer, String, Date, Time, Numeric, Boolean, ForeignKey
from sqlalchemy.orm import declarative_base, relationship

Base = declarative_base()

class Customer(Base):
    __tablename__ = "customer"

    customer_id = Column(Integer, primary_key=True, autoincrement=True)
    customer_code = Column(String(100), unique=True, nullable=False)
    customer_name = Column(String(100), nullable=False)
    gender = Column(String(10), nullable=False)
    age = Column(Integer, nullable=False)
    customer_contact = Column(String(30), nullable=False)
    customer_email = Column(String(100), nullable=False)

    accounts = relationship("Account", back_populates="customer")


class State(Base):
    __tablename__ = "state"

    state_id = Column(Integer, primary_key=True, autoincrement=True)
    state_name = Column(String(100), unique=True, nullable=False)

    cities = relationship("City", back_populates="state")


class City(Base):
    __tablename__ = "city"

    city_id = Column(Integer, primary_key=True, autoincrement=True)
    city_name = Column(String(100), nullable=False)
    state_id = Column(Integer, ForeignKey("state.state_id"), nullable=False)

    state = relationship("State", back_populates="cities")
    bank_branches = relationship("BankBranch", back_populates="city")


class BankBranch(Base):
    __tablename__ = "bank_branch"

    bank_branch_id = Column(Integer, primary_key=True, autoincrement=True)
    bank_branch_name = Column(String(100), nullable=False)
    city_id = Column(Integer, ForeignKey("city.city_id"), nullable=False)


class AccountType(Base):
    __tablename__ = "account_type"

    account_type_id = Column(Integer, primary_key=True, autoincrement=True)
    account_type_name = Column(String(50), unique=True, nullable=False)

    accounts = relationship("Account", back_populates="account_type")


class Account(Base):
    __tablename__ = "account"

    account_id = Column(Integer, primary_key=True, autoincrement=True)
    customer_id = Column(Integer, ForeignKey("customer.customer_id"), nullable=False)
    account_type_id = Column(Integer, ForeignKey("account_type.account_type_id"), nullable=False)
    bank_branch_id = Column(Integer, ForeignKey("bank_branch.bank_branch_id"), nullable=False)

    customer = relationship("Customer", back_populates="accounts")
    account_type = relationship("AccountType", back_populates="accounts")
    bank_branch = relationship("BankBranch", back_populates="accounts")
    transactions = relationship("BankTransaction", back_populates="account")


class TransactionType(Base):
    __tablename__ = "transaction_type"

    transaction_type_id = Column(Integer, primary_key=True, autoincrement=True)
    transaction_type_name = Column(String(50), unique=True, nullable=False)

    transactions = relationship("BankTransaction", back_populates="transaction_type")


class MerchantCategory(Base):
    __tablename__ = "merchant_category"

    merchant_category_id = Column(Integer, primary_key=True, autoincrement=True)
    merchant_category_name = Column(String(100), unique=True, nullable=False)

    merchants = relationship("Merchant", back_populates="merchant_category")


class Merchant(Base):
    __tablename__ = "merchant"

    merchant_id = Column(Integer, primary_key=True, autoincrement=True)
    merchant_code = Column(String(100), unique=True, nullable=False)
    merchant_category_id = Column(Integer, ForeignKey("merchant_category.merchant_category_id"), nullable=False)

    merchant_category = relationship("MerchantCategory", back_populates="merchants")
    transactions = relationship("BankTransaction", back_populates="merchant")


class DeviceType(Base):
    __tablename__ = "device_type"

    device_type_id = Column(Integer, primary_key=True, autoincrement=True)
    device_type_name = Column(String(50), unique=True, nullable=False)

    transaction_devices = relationship("TransactionDevice", back_populates="device_type")


class TransactionDevice(Base):
    __tablename__ = "transaction_device"

    transaction_device_id = Column(Integer, primary_key=True, autoincrement=True)
    transaction_device_name = Column(String(100), unique=True, nullable=False)
    device_type_id = Column(Integer, ForeignKey("device_type.device_type_id"), nullable=False)

    device_type = relationship("DeviceType", back_populates="transaction_devices")
    transactions = relationship("BankTransaction", back_populates="transaction_device")


class TransactionLocation(Base):
    __tablename__ = "transaction_location"

    transaction_location_id = Column(Integer, primary_key=True, autoincrement=True)
    transaction_location_name = Column(String(150), unique=True, nullable=False)

    transactions = relationship("BankTransaction", back_populates="transaction_location")


class Currency(Base):
    __tablename__ = "currency"

    currency_id = Column(Integer, primary_key=True, autoincrement=True)
    currency_code = Column(String(10), unique=True, nullable=False)

    transactions = relationship("BankTransaction", back_populates="currency")


class BankTransaction(Base):
    __tablename__ = "bank_transaction"

    transaction_id = Column(Integer, primary_key=True, autoincrement=True)
    transaction_code = Column(String(100), unique=True, nullable=False)
    account_id = Column(Integer, ForeignKey("account.account_id"), nullable=False)
    merchant_id = Column(Integer, ForeignKey("merchant.merchant_id"), nullable=False)
    transaction_type_id = Column(Integer, ForeignKey("transaction_type.transaction_type_id"), nullable=False)
    transaction_device_id = Column(Integer, ForeignKey("transaction_device.transaction_device_id"), nullable=False)
    transaction_location_id = Column(Integer, ForeignKey("transaction_location.transaction_location_id"), nullable=False)
    currency_id = Column(Integer, ForeignKey("currency.currency_id"), nullable=False)
    transaction_date = Column(Date, nullable=False)
    transaction_time = Column(Time, nullable=False)
    transaction_amount = Column(Numeric(12, 2), nullable=False)
    account_balance = Column(Numeric(12, 2), nullable=False)
    is_fraud = Column(Boolean, nullable=False)
    transaction_description = Column(String(255), nullable=False)

    account = relationship("Account", back_populates="transactions")
    merchant = relationship("Merchant", back_populates="transactions")
    transaction_type = relationship("TransactionType", back_populates="transactions")
    transaction_device = relationship("TransactionDevice", back_populates="transactions")
    transaction_location = relationship("TransactionLocation", back_populates="transactions")
    currency = relationship("Currency", back_populates="transactions")

In [26]:
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

print("Tablice kreirane.")

Tablice kreirane.


In [28]:
with engine.connect() as connection:
    result = connection.execute(text("SHOW TABLES;"))
    for row in result:
        print(row[0])

account
account_type
bank_branch
bank_transaction
city
currency
customer
device_type
merchant
merchant_category
state
transaction_device
transaction_location
transaction_type


### 4.3. EER dijagram

<img src="EER_dijagram.png" alt="EER dijagram - logički model" width="700">

## 5. Učitavanje podataka u relacijsku bazu

### 5.1. Priprema podataka za unos u bazu

In [30]:
print(df80["Transaction_Time"].head(20))

0     16:04:07
2     03:09:52
5     06:49:53
6     00:53:33
7     04:04:48
8     10:50:12
9     04:37:34
10    08:52:44
11    19:43:31
12    22:02:12
15    09:50:59
16    13:51:31
17    12:41:39
18    12:42:43
19    21:50:44
20    08:34:34
22    07:57:53
23    09:35:39
25    12:10:24
26    04:13:12
Name: Transaction_Time, dtype: object


In [32]:
df80 = df80.copy()

df80["Transaction_Date"] = pd.to_datetime(df80["Transaction_Date"]).dt.date

df80["Transaction_Time"] = pd.to_datetime(
    df80["Transaction_Time"].astype(str),
    format="%H:%M:%S",
    errors="coerce"
).dt.time

print(df80.shape)
print(df80[["Transaction_Date", "Transaction_Time"]].isnull().sum())

(160000, 24)
Transaction_Date    0
Transaction_Time    0
dtype: int64


### 5.2. Resetiranje relacijskih tablica

In [34]:
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

print("Tablice resetirane i ponovno kreirane.")

Tablice resetirane i ponovno kreirane.


### 5.3. Popunjavanje pomoćnih tablica

In [36]:
state_df = df80[["State"]].drop_duplicates().rename(columns={"State": "state_name"})
state_df.to_sql("state", engine, if_exists="append", index=False)

account_type_df = df80[["Account_Type"]].drop_duplicates().rename(columns={"Account_Type": "account_type_name"})
account_type_df.to_sql("account_type", engine, if_exists="append", index=False)

transaction_type_df = df80[["Transaction_Type"]].drop_duplicates().rename(columns={"Transaction_Type": "transaction_type_name"})
transaction_type_df.to_sql("transaction_type", engine, if_exists="append", index=False)

merchant_category_df = df80[["Merchant_Category"]].drop_duplicates().rename(columns={"Merchant_Category": "merchant_category_name"})
merchant_category_df.to_sql("merchant_category", engine, if_exists="append", index=False)

device_type_df = df80[["Device_Type"]].drop_duplicates().rename(columns={"Device_Type": "device_type_name"})
device_type_df.to_sql("device_type", engine, if_exists="append", index=False)

transaction_location_df = df80[["Transaction_Location"]].drop_duplicates().rename(columns={"Transaction_Location": "transaction_location_name"})
transaction_location_df.to_sql("transaction_location", engine, if_exists="append", index=False)

currency_df = df80[["Transaction_Currency"]].drop_duplicates().rename(columns={"Transaction_Currency": "currency_code"})
currency_df.to_sql("currency", engine, if_exists="append", index=False)

print("Pomoćne tablice su popunjene.")

Pomoćne tablice su popunjene.


In [38]:
tables = [
    "state",
    "account_type",
    "transaction_type",
    "merchant_category",
    "device_type",
    "transaction_location",
    "currency"
]

with engine.connect() as connection:
    for table in tables:
        count = connection.execute(text(f"SELECT COUNT(*) FROM {table};")).fetchone()[0]
        print(table, ":", count)

state : 34
account_type : 3
transaction_type : 5
merchant_category : 6
device_type : 4
transaction_location : 148
currency : 1


### 5.4. Popunjavanje tablica city i bank_branch

In [40]:
state_map = pd.read_sql("SELECT state_id, state_name FROM state", engine)
state_map = state_map.set_index("state_name")["state_id"]

city_df = df80[["City", "State"]].drop_duplicates()
city_df["state_id"] = city_df["State"].map(state_map)

city_df = city_df[["City", "state_id"]].rename(columns={
    "City": "city_name"
})

city_df.to_sql("city", engine, if_exists="append", index=False)

city_db = pd.read_sql("SELECT city_id, city_name, state_id FROM city", engine)

bank_branch_df = df80[["Bank_Branch", "City", "State"]].drop_duplicates()
bank_branch_df["state_id"] = bank_branch_df["State"].map(state_map)

bank_branch_df = bank_branch_df.merge(
    city_db,
    left_on=["City", "state_id"],
    right_on=["city_name", "state_id"],
    how="left"
)

bank_branch_df = bank_branch_df[["Bank_Branch", "city_id"]].rename(columns={
    "Bank_Branch": "bank_branch_name"
})

bank_branch_df = bank_branch_df.drop_duplicates(subset=["bank_branch_name", "city_id"])

bank_branch_df.to_sql("bank_branch", engine, if_exists="append", index=False)

print("Tablice city i bank_branch su popunjene.")

Tablice city i bank_branch su popunjene.


In [42]:
tables = ["city", "bank_branch"]

with engine.connect() as connection:
    for table in tables:
        count = connection.execute(text(f"SELECT COUNT(*) FROM {table};")).fetchone()[0]
        print(table, ":", count)

city : 148
bank_branch : 148


### 5.5. Popunjavanje glavnih povezanih tablica

In [44]:
customer_df = df80[
    ["Customer_ID", "Customer_Name", "Gender", "Age", "Customer_Contact", "Customer_Email"]
].drop_duplicates(subset=["Customer_ID"])

customer_df = customer_df.rename(columns={
    "Customer_ID": "customer_code",
    "Customer_Name": "customer_name",
    "Gender": "gender",
    "Age": "age",
    "Customer_Contact": "customer_contact",
    "Customer_Email": "customer_email"
})

customer_df.to_sql("customer", engine, if_exists="append", index=False)


merchant_category_map = pd.read_sql(
    "SELECT merchant_category_id, merchant_category_name FROM merchant_category",
    engine
)
merchant_category_map = merchant_category_map.set_index("merchant_category_name")["merchant_category_id"]

merchant_df = df80[["Merchant_ID", "Merchant_Category"]].drop_duplicates(subset=["Merchant_ID"])
merchant_df["merchant_category_id"] = merchant_df["Merchant_Category"].map(merchant_category_map)

merchant_df = merchant_df[["Merchant_ID", "merchant_category_id"]].rename(columns={
    "Merchant_ID": "merchant_code"
})

merchant_df.to_sql("merchant", engine, if_exists="append", index=False)


device_type_map = pd.read_sql(
    "SELECT device_type_id, device_type_name FROM device_type",
    engine
)
device_type_map = device_type_map.set_index("device_type_name")["device_type_id"]

transaction_device_df = df80[["Transaction_Device", "Device_Type"]].drop_duplicates(subset=["Transaction_Device"])
transaction_device_df["device_type_id"] = transaction_device_df["Device_Type"].map(device_type_map)

transaction_device_df = transaction_device_df[["Transaction_Device", "device_type_id"]].rename(columns={
    "Transaction_Device": "transaction_device_name"
})

transaction_device_df.to_sql("transaction_device", engine, if_exists="append", index=False)


customer_map = pd.read_sql(
    "SELECT customer_id, customer_code FROM customer",
    engine
)
customer_map = customer_map.set_index("customer_code")["customer_id"]

account_type_map = pd.read_sql(
    "SELECT account_type_id, account_type_name FROM account_type",
    engine
)
account_type_map = account_type_map.set_index("account_type_name")["account_type_id"]

state_map = pd.read_sql(
    "SELECT state_id, state_name FROM state",
    engine
)
state_map = state_map.set_index("state_name")["state_id"]

city_db = pd.read_sql(
    "SELECT city_id, city_name, state_id FROM city",
    engine
)

bank_branch_db = pd.read_sql(
    "SELECT bank_branch_id, bank_branch_name, city_id FROM bank_branch",
    engine
)

account_df = df80[["Customer_ID", "Account_Type", "Bank_Branch", "City", "State"]].drop_duplicates()

account_df["customer_id"] = account_df["Customer_ID"].map(customer_map)
account_df["account_type_id"] = account_df["Account_Type"].map(account_type_map)
account_df["state_id"] = account_df["State"].map(state_map)

account_df = account_df.merge(
    city_db,
    left_on=["City", "state_id"],
    right_on=["city_name", "state_id"],
    how="left"
)

account_df = account_df.merge(
    bank_branch_db,
    left_on=["Bank_Branch", "city_id"],
    right_on=["bank_branch_name", "city_id"],
    how="left"
)

account_df = account_df[["customer_id", "account_type_id", "bank_branch_id"]]
account_df = account_df.drop_duplicates()

print(account_df.isnull().sum())

account_df.to_sql("account", engine, if_exists="append", index=False)

print("Tablice customer, merchant, transaction_device i account su popunjene.")

customer_id        0
account_type_id    0
bank_branch_id     0
dtype: int64
Tablice customer, merchant, transaction_device i account su popunjene.


In [46]:
tables = ["customer", "merchant", "transaction_device", "account"]

with engine.connect() as connection:
    for table in tables:
        count = connection.execute(text(f"SELECT COUNT(*) FROM {table};")).fetchone()[0]
        print(table, ":", count)

customer : 160000
merchant : 160000
transaction_device : 20
account : 160000


### 5.6. Popunjavanje tablice bank_transaction

In [48]:
account_db = pd.read_sql(
    "SELECT account_id, customer_id, account_type_id, bank_branch_id FROM account",
    engine
)

transaction_type_map = pd.read_sql(
    "SELECT transaction_type_id, transaction_type_name FROM transaction_type",
    engine
)
transaction_type_map = transaction_type_map.set_index("transaction_type_name")["transaction_type_id"]

merchant_map = pd.read_sql(
    "SELECT merchant_id, merchant_code FROM merchant",
    engine
)
merchant_map = merchant_map.set_index("merchant_code")["merchant_id"]

transaction_device_map = pd.read_sql(
    "SELECT transaction_device_id, transaction_device_name FROM transaction_device",
    engine
)
transaction_device_map = transaction_device_map.set_index("transaction_device_name")["transaction_device_id"]

transaction_location_map = pd.read_sql(
    "SELECT transaction_location_id, transaction_location_name FROM transaction_location",
    engine
)
transaction_location_map = transaction_location_map.set_index("transaction_location_name")["transaction_location_id"]

currency_map = pd.read_sql(
    "SELECT currency_id, currency_code FROM currency",
    engine
)
currency_map = currency_map.set_index("currency_code")["currency_id"]


transaction_df = df80.copy()

transaction_df["customer_id"] = transaction_df["Customer_ID"].map(customer_map)
transaction_df["account_type_id"] = transaction_df["Account_Type"].map(account_type_map)
transaction_df["state_id"] = transaction_df["State"].map(state_map)

transaction_df = transaction_df.merge(
    city_db,
    left_on=["City", "state_id"],
    right_on=["city_name", "state_id"],
    how="left"
)

transaction_df = transaction_df.merge(
    bank_branch_db,
    left_on=["Bank_Branch", "city_id"],
    right_on=["bank_branch_name", "city_id"],
    how="left"
)

transaction_df = transaction_df.merge(
    account_db,
    on=["customer_id", "account_type_id", "bank_branch_id"],
    how="left"
)

transaction_df["merchant_id"] = transaction_df["Merchant_ID"].map(merchant_map)
transaction_df["transaction_type_id"] = transaction_df["Transaction_Type"].map(transaction_type_map)
transaction_df["transaction_device_id"] = transaction_df["Transaction_Device"].map(transaction_device_map)
transaction_df["transaction_location_id"] = transaction_df["Transaction_Location"].map(transaction_location_map)
transaction_df["currency_id"] = transaction_df["Transaction_Currency"].map(currency_map)

bank_transaction_df = transaction_df.rename(columns={
    "Transaction_ID": "transaction_code",
    "Transaction_Date": "transaction_date",
    "Transaction_Time": "transaction_time",
    "Transaction_Amount": "transaction_amount",
    "Account_Balance": "account_balance",
    "Is_Fraud": "is_fraud",
    "Transaction_Description": "transaction_description"
})

bank_transaction_df = bank_transaction_df[
    [
        "transaction_code",
        "account_id",
        "merchant_id",
        "transaction_type_id",
        "transaction_device_id",
        "transaction_location_id",
        "currency_id",
        "transaction_date",
        "transaction_time",
        "transaction_amount",
        "account_balance",
        "is_fraud",
        "transaction_description"
    ]
]

print(bank_transaction_df.shape)
print(bank_transaction_df.isnull().sum())

(160000, 13)
transaction_code           0
account_id                 0
merchant_id                0
transaction_type_id        0
transaction_device_id      0
transaction_location_id    0
currency_id                0
transaction_date           0
transaction_time           0
transaction_amount         0
account_balance            0
is_fraud                   0
transaction_description    0
dtype: int64


In [49]:
bank_transaction_df.to_sql(
    "bank_transaction",
    engine,
    if_exists="append",
    index=False,
    chunksize=5000
)

print("Tablica bank_transaction je popunjena.")

Tablica bank_transaction je popunjena.


In [52]:
with engine.connect() as connection:
    count = connection.execute(text("SELECT COUNT(*) FROM bank_transaction;")).fetchone()[0]
    print("bank_transaction:", count)

bank_transaction: 160000


### 5.7. Provjera broja učitanih zapisa

In [54]:
tables = [
    "state",
    "city",
    "bank_branch",
    "account_type",
    "customer",
    "account",
    "transaction_type",
    "merchant_category",
    "merchant",
    "device_type",
    "transaction_device",
    "transaction_location",
    "currency",
    "bank_transaction"
]

with engine.connect() as connection:
    for table in tables:
        count = connection.execute(text(f"SELECT COUNT(*) FROM {table};")).fetchone()[0]
        print(table, ":", count)

state : 34
city : 148
bank_branch : 148
account_type : 3
customer : 160000
account : 160000
transaction_type : 5
merchant_category : 6
merchant : 160000
device_type : 4
transaction_device : 20
transaction_location : 148
currency : 1
bank_transaction : 160000


### 5.8. Testni SELECT upit

Provjeravam jesu li podaci ispravno povezani između glavne transakcijske tablice i pomoćnih tablica.

In [56]:
query = """
SELECT 
    bt.transaction_code,
    c.customer_name,
    at.account_type_name,
    bb.bank_branch_name,
    tt.transaction_type_name,
    mc.merchant_category_name,
    td.transaction_device_name,
    tl.transaction_location_name,
    cur.currency_code,
    bt.transaction_amount,
    bt.account_balance,
    bt.is_fraud
FROM bank_transaction bt
JOIN account a ON bt.account_id = a.account_id
JOIN customer c ON a.customer_id = c.customer_id
JOIN account_type at ON a.account_type_id = at.account_type_id
JOIN bank_branch bb ON a.bank_branch_id = bb.bank_branch_id
JOIN transaction_type tt ON bt.transaction_type_id = tt.transaction_type_id
JOIN merchant m ON bt.merchant_id = m.merchant_id
JOIN merchant_category mc ON m.merchant_category_id = mc.merchant_category_id
JOIN transaction_device td ON bt.transaction_device_id = td.transaction_device_id
JOIN transaction_location tl ON bt.transaction_location_id = tl.transaction_location_id
JOIN currency cur ON bt.currency_id = cur.currency_id
LIMIT 10;
"""

test_df = pd.read_sql(query, engine)
test_df

,transaction_code,customer_name,account_type_name,bank_branch_name,transaction_type_name,merchant_category_name,transaction_device_name,transaction_location_name,currency_code,transaction_amount,account_balance,is_fraud
0,c18a6b44-04ac-49e6-be01-a89cd371ba05,Nandini Dugar,Business,Faridabad Branch,Bill Payment,Restaurant,Mobile Device,"Faridabad, Haryana",INR,6155.84,85128.55,0
1,2c0cacab-96c1-4c56-82fc-6abe4f55809d,Riya Bandi,Business,Lunglei Branch,Withdrawal,Clothing,Self-service Banking Machine,"Lunglei, Mizoram",INR,19668.24,59128.53,0
2,695a5d36-a9ab-4f70-bdac-1800053f50bf,Veer Sami,Business,Mapusa Branch,Debit,Clothing,Wearable Device,"Mapusa, Goa",INR,71876.95,70421.33,0
3,f03b0282-a8a0-4f2b-a41b-ed9e8a8e4215,Abhiram Gole,Business,New Delhi Branch,Withdrawal,Groceries,ATM Booth Kiosk,"New Delhi, Delhi",INR,66555.24,16218.12,0
4,933f36c9-b5bb-4f93-8850-c9a344921edb,Isaiah Sarraf,Business,Thiruvananthapuram Branch,Bill Payment,Restaurant,POS Terminal,"Thiruvananthapuram, Kerala",INR,26620.04,49573.58,0
5,64425200-7cf3-4cdc-b3de-93645ce54960,Nitara Taneja,Business,Diu Branch,Debit,Health,Mobile Device,"Diu, Dadra and Nagar Haveli and Daman and Diu",INR,35527.87,45371.19,0
6,98d0f0d8-2990-47f6-b8cf-dc5bbc508f65,Fariq Sama,Business,Pune Branch,Withdrawal,Health,ATM Booth Kiosk,"Pune, Maharashtra",INR,56997.16,52570.81,0
7,6867597b-c2d7-448f-837b-2b56d27788b7,Balvan Mital,Business,Naharlagun Branch,Withdrawal,Clothing,Self-service Banking Machine,"Naharlagun, Arunachal Pradesh",INR,87361.82,43663.73,0
8,ecb0450a-5b10-4e41-a59c-29c467ba354b,Diya Thakkar,Business,Jagdalpur Branch,Withdrawal,Groceries,ATM,"Jagdalpur, Chhattisgarh",INR,59774.14,40678.87,1
9,069633e1-d694-4b3d-8416-bad9f2141fa5,Jacob Uppal,Business,Jorethang Branch,Transfer,Electronics,ATM,"Jorethang, Sikkim",INR,70813.29,20921.72,0
